In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [2]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, 10)
        # self.fc1 = nn.Linear(hidden_size, 10)
        # self.act = nn.ReLU()

    def forward(self, x):
        x = self.fc0(x)
        return x

In [3]:
hyperparams = {
    "num_epochs": 32,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()), # replace to default
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.4},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.05,
    "min_epochs_between_expansions": 8,
    # "min_epochs_between_expansions": 10_000,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 2000,
    "weight_decay": 1e-4,
    "diff_threshold": 0.05
}

In [4]:
# Dataset and Dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

# Load dataset and split into train/validation sets
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

train_dataset, val_dataset = random_split(dataset, [0.8, 0.2])

train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=hyperparams['batch_size'], shuffle=False)

In [5]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [6]:
# import wandb
# wandb.login()

In [7]:
# wandb.finish()
# run = wandb.init(
#     project="self-expanding-nets",
#     name=f"name",
#     config=hyperparams
# )

In [8]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="mnist",
)s
experiment.log_parameters(hyperparams)
experiment.set_name("trash")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/vanya9090/general/22d24d950a234c7f9d89a89325beced0



In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, train_loader, val_loader, criterion,optimizer, hyperparams, device, experiment)

100%|██████████| 188/188 [00:03<00:00, 54.94it/s]
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1262372 > 1048576). Falling back to PNG format.



Class      Accuracy   Samples   
------------------------------
0          0.9250    1226      
1          0.9387    1354      
2          0.8545    1155      
3          0.8534    1194      
4          0.8737    1180      
5          0.7344    1122      
6          0.9195    1192      
7          0.8647    1286      
8          0.8341    1151      
9          0.8798    1140      
------------------------------
Average    0.8678
Overall    0.8698
Epoch 1/32, Train Loss: 0.9315, Val Loss: 0.5400, Val Accuracy: 0.8698


100%|██████████| 188/188 [00:02<00:00, 70.76it/s]
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1270272 > 1048576). Falling back to PNG format.



Class      Accuracy   Samples   
------------------------------
0          0.9413    1226      
1          0.9638    1354      
2          0.8494    1155      
3          0.8710    1194      
4          0.9093    1180      
5          0.8200    1122      
6          0.9446    1192      
7          0.8935    1286      
8          0.8454    1151      
9          0.8772    1140      
------------------------------
Average    0.8915
Overall    0.8934
Epoch 2/32, Train Loss: 0.4607, Val Loss: 0.4130, Val Accuracy: 0.8934


100%|██████████| 188/188 [00:02<00:00, 82.95it/s]
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1272963 > 1048576). Falling back to PNG format.



Class      Accuracy   Samples   
------------------------------
0          0.9437    1226      
1          0.9653    1354      
2          0.8684    1155      
3          0.8735    1194      
4          0.9085    1180      
5          0.8316    1122      
6          0.9505    1192      
7          0.9036    1286      
8          0.8671    1151      
9          0.8939    1140      
------------------------------
Average    0.9006
Overall    0.9022
Epoch 3/32, Train Loss: 0.3836, Val Loss: 0.3664, Val Accuracy: 0.9022


... (35 outputs left out)


Class      Accuracy   Samples   
------------------------------
0          0.9935    1226      
1          0.9970    1354      
2          0.9593    1155      
3          0.9807    1194      
4          0.9780    1180      
5          0.9528    1122      
6          0.9950    1192      
7          0.9774    1286      
8          0.9696    1151      
9          0.9711    1140      
------------------------------
Average    0.9774
Overall    0.9780
Epoch 21/32, Train Loss: 0.0237, Val Loss: 0.0790, Val Accuracy: 0.9780


 61%|██████    | 114/188 [00:19<00:12,  5.75it/s]


KeyboardInterrupt: 

COMET ERROR: Due to connectivity issues, there's an error in processing the heartbeat. The experiment's status updates might be inaccurate until the connection issues are resolved.


In [10]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : vast_rafter_684
COMET INFO:     url                   : https://www.comet.com/vanya9090/general/2d3810dbd31444b6ad0fa977d5000bb8
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     acc amount [32]               : (4.5100058858151854e-06, 0.00011698554421768707)
COMET INFO:     len_choose [3]                : (49, 845)
COMET INFO:     lr                            : 0.001
COMET INFO:     n_params over train_time [32] : (2980.92171642659, 90506.52486510566)
COMET INFO:     params amount [32]            : (7840, 213146)
COMET INFO:     params ratio [32]             : (0.8808673469387756, 0.9999577754215421)
COMET INFO:     params to replace amount

In [11]:
torch.save(sparse_model, 'mnist.pt')